
# 스마트팜 원시데이터 가설검증 노트북 (비판적 검토 포함)

이 노트북은 **결과를 그대로 믿지 않는다**는 전제로 작성되었습니다.

핵심 원칙:
- p-value 하나만으로 결론 내리지 않습니다.
- 시계열 자기상관이 큰 원시 1분 데이터는 그대로 독립표본처럼 해석하지 않습니다.
- **기술통계는 raw 기준**, **가설검정은 10분 집계본 기준**으로 수행합니다.
- 두 집단/다중 집단 검정은 **외생 그룹 라벨이 있을 때만** 실행합니다.
- 파생변수와 원재료 변수의 상관은 **구조적 상관**과 **경험적 상관**으로 분리합니다.

이번 제출 구조:
1. 정규성 검정 (Shapiro-Wilk, K-S 참고)
2. 두 집단 비교 (t-test, Mann-Whitney U) — 외생 라벨 있을 때만
3. 다중 집단 비교 (ANOVA, Kruskal-Wallis) — 외생 라벨 있을 때만
4. 상관/독립성 (Pearson, Spearman, 카이제곱)
5. 분포 변화 감지 (KS drift test)
6. 비판적 리뷰 체크리스트


In [1]:

import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.unicode_minus"] = False


In [2]:

# ------------------------------
# 사용자 설정
# ------------------------------
RESAMPLE_RULE = "10min"      # 가설검정용 집계 단위. row-level 검정보다 안전함.
RANDOM_STATE = 42
SHAPIRO_MAX_N = 5000         # Shapiro는 전체가 너무 크면 과민하므로 샘플링
KS_MAX_N = 5000              # K-S 정규성 확인용 샘플 크기
MIN_GROUP_N = 20             # 집단 검정 최소 표본 수

# Drift 판정은 p-value 하나로 끝내지 않음.
DRIFT_ALPHA = 0.05
DRIFT_STAT_WEAK = 0.05
DRIFT_STAT_STRONG = 0.10
DRIFT_MEDIAN_WEAK = 0.05     # 5%
DRIFT_MEDIAN_STRONG = 0.10   # 10%

# 외생 그룹 후보: 실제 CSV에 있으면 자동 사용, 없으면 자동 스킵.
EXTERNAL_GROUP_CANDIDATES = [
    "cleaning_event_flag",
    "hidden_tip_clog_level",
    "hidden_blocked_tip_ratio",
    "hidden_risk_stage",
    "anomaly_label",
    "root_cause_label",
]

# 피처 설계 근거용 상관 분석 대상
TARGET_DICTIONARY = {
    "motor_current_a": ["motor_power_kw", "motor_temperature_c", "pump_rpm", "discharge_pressure_kpa"],
    "zone1_resistance": ["zone1_pressure_kpa", "zone1_flow_l_min", "filter_delta_p_kpa", "pump_rpm"],
    "mix_ph": ["dosing_acid_ml_min", "mix_target_ph", "mix_temp_c", "mix_flow_l_min", "acid_tank_level_pct"],
    "flow_rate_l_min": ["pump_rpm", "suction_pressure_kpa", "discharge_pressure_kpa", "filter_delta_p_kpa"],
}
PRIMARY_TARGETS = list(TARGET_DICTIONARY.keys())

# 파생변수의 정의식 구성요소를 분리해서 '구조적 상관'으로 표기
STRUCTURAL_RELATIONSHIPS = {
    "zone1_resistance": {"zone1_pressure_kpa", "zone1_flow_l_min"},
    "filter_delta_p_kpa": {"filter_pressure_in_kpa", "filter_pressure_out_kpa"},
    "differential_pressure_kpa": {"discharge_pressure_kpa", "suction_pressure_kpa"},
    "ec_error_ds_m": {"mix_ec_ds_m", "mix_target_ec_ds_m"},
    "ph_error": {"mix_ph", "mix_target_ph"},
}


In [3]:

# ------------------------------
# 경로 탐색 및 데이터 로드
# ------------------------------
def resolve_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(f"파일을 찾지 못했습니다: {paths}")

DATA_PATH = resolve_first_existing([
    "selected_smartfarm.csv",
    "./selected_smartfarm.csv",
    "/mnt/data/selected_smartfarm.csv",
    "src/selected_smartfarm.csv",
    "human_A/src/selected_smartfarm.csv",
])

COLINFO_PATH = resolve_first_existing([
    "smartfarm_columns.xlsx",
    "./smartfarm_columns.xlsx",
    "/mnt/data/smartfarm_columns.xlsx",
])

print("DATA_PATH:", DATA_PATH)
print("COLINFO_PATH:", COLINFO_PATH)

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
col_df = pd.read_excel(COLINFO_PATH)

col_desc = {}
if {"컬럼명", "설명"}.issubset(col_df.columns):
    col_desc = dict(col_df[["컬럼명", "설명"]].dropna().itertuples(index=False, name=None))

# ------------------------------
# 파생변수 생성
# ------------------------------
# 0으로 나누는 문제는 NaN 처리. 임의 clip은 하지 않음.
df["filter_delta_p_kpa"] = df["filter_pressure_in_kpa"] - df["filter_pressure_out_kpa"]
df["differential_pressure_kpa"] = df["discharge_pressure_kpa"] - df["suction_pressure_kpa"]
df["zone1_resistance"] = np.where(
    df["zone1_flow_l_min"].abs() > 1e-6,
    df["zone1_pressure_kpa"] / df["zone1_flow_l_min"],
    np.nan,
)
df["ec_error_ds_m"] = df["mix_ec_ds_m"] - df["mix_target_ec_ds_m"]
df["ph_error"] = df["mix_ph"] - df["mix_target_ph"]

# ------------------------------
# 가설검정용 집계 데이터 생성
# ------------------------------
def mode_or_nan(x):
    m = x.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
num_resampled = df.set_index("timestamp")[numeric_cols].resample(RESAMPLE_RULE).median()

available_group_cols_raw = [c for c in EXTERNAL_GROUP_CANDIDATES if c in df.columns]
if available_group_cols_raw:
    group_resampled = df.set_index("timestamp")[available_group_cols_raw].resample(RESAMPLE_RULE).agg(mode_or_nan)
    infer_df = pd.concat([num_resampled, group_resampled], axis=1).reset_index()
else:
    infer_df = num_resampled.reset_index()

report_cols = [c for c in dict.fromkeys(sum(([k] + v for k, v in TARGET_DICTIONARY.items()), [])) if c in infer_df.columns]

print(f"raw shape      : {df.shape}")
print(f"infer shape    : {infer_df.shape}  (rule={RESAMPLE_RULE})")
print(f"report columns : {len(report_cols)}개")
print(f"external groups: {available_group_cols_raw if available_group_cols_raw else '없음'}")


DATA_PATH: selected_smartfarm.csv
COLINFO_PATH: smartfarm_columns.xlsx


raw shape      : (129591, 50)
infer shape    : (12960, 50)  (rule=10min)
report columns : 17개
external groups: 없음



## 0. 데이터 개요
- 기술통계와 결측치는 raw 데이터 기준으로 확인합니다.
- 실제 가설검정은 10분 집계본(`infer_df`)을 기준으로 수행합니다.
- 이유: 원시 1분 데이터는 자기상관이 너무 높아 p-value가 과도하게 작아질 수 있기 때문입니다.


In [4]:

overview = pd.DataFrame({
    "metric": [
        "raw_rows", "raw_cols", "start_time", "end_time", "missing_rows_any",
        "report_target_count", "external_group_count"
    ],
    "value": [
        len(df), df.shape[1], df["timestamp"].min(), df["timestamp"].max(), int(df.isna().any(axis=1).sum()),
        len(report_cols), len(available_group_cols_raw)
    ]
})

missing_top = (
    df.isna().mean().mul(100).sort_values(ascending=False).reset_index()
      .rename(columns={"index": "variable", 0: "missing_pct"})
)
missing_top["description"] = missing_top["variable"].map(col_desc)

print("[Overview]")
display(overview)
print("[Top missing variables - raw 기준]")
display(missing_top.head(15))


[Overview]


,metric,value
0,raw_rows,129591
1,raw_cols,50
2,start_time,2026-03-01 00:00:00
3,end_time,2026-05-29 23:50:00
4,missing_rows_any,871
5,report_target_count,17
6,external_group_count,0


[Top missing variables - raw 기준]


,variable,missing_pct,description
0,zone1_resistance,0.6721,NaN
1,timestamp,0.0000,데이터 측정 시각
2,air_temp_c,0.0000,재배 공간 내부 공기 온도
3,relative_humidity_pct,0.0000,실내 상대 습도
4,co2_ppm,0.0000,실내 CO₂ 농도
5,raw_tank_level_pct,0.0000,원수 저장 탱크 수위
6,raw_water_temp_c,0.0000,원수 탱크 내 물 온도
7,pump_rpm,0.0000,펌프 모터 회전 속도
8,flow_rate_l_min,0.0000,실제 토출 유량
9,suction_pressure_kpa,0.0000,펌프 흡입측 압력



## 1. 가정 점검: 독립성(시계열 자기상관)
이 단계는 사용자가 요구한 **비판적 리뷰**의 핵심입니다.

해석 원칙:
- lag-1 자기상관이 높을수록 인접 행이 서로 비슷합니다.
- 이런 데이터에 row-level t-test/MWU/KS를 바로 쓰면 p-value가 과소추정될 수 있습니다.
- 따라서 **raw는 설명용**, **집계본은 검정용**으로 분리합니다.


In [5]:

def lag1_autocorr(series):
    s = pd.Series(series).dropna()
    return s.autocorr(lag=1) if len(s) > 2 else np.nan

autocorr_df = pd.DataFrame([
    {
        "variable": c,
        "description": col_desc.get(c, ""),
        "lag1_raw": lag1_autocorr(df[c]),
        "lag1_resampled": lag1_autocorr(infer_df[c]),
    }
    for c in PRIMARY_TARGETS if c in infer_df.columns
])

autocorr_df["review"] = np.where(
    autocorr_df["lag1_resampled"] > 0.8,
    "주의: 집계 후에도 자기상관 큼",
    "상대적으로 완화됨"
)

display(autocorr_df)


,variable,description,lag1_raw,lag1_resampled,review
0,motor_current_a,모터 소비 전류,0.9997,0.9860,주의: 집계 후에도 자기상관 큼
1,zone1_resistance,,0.8616,0.4475,상대적으로 완화됨
2,mix_ph,실제 pH 값,0.9821,0.4882,상대적으로 완화됨
3,flow_rate_l_min,실제 토출 유량,0.9999,0.9922,주의: 집계 후에도 자기상관 큼



## 2. 정규성 검정
실무 해석 규칙:
- **Shapiro-Wilk**: 민감한 검정이므로 대용량에서는 샘플링해서 참고합니다.
- **K-S 정규성 검정**: 평균/표준편차를 같은 데이터에서 추정하므로 엄밀한 최종판정용이 아니라 **보조 참고값**으로만 씁니다.
- **왜도(skew)**가 크면 비모수 검정을 우선 추천합니다.

최종 의사결정:
- `|skew| >= 1` 또는 `Shapiro p < 0.05`이면 **비모수 권장**
- 그 외는 **정규 근사 가능**으로만 둡니다.


In [6]:

def normality_summary(series, shapiro_max_n=SHAPIRO_MAX_N, ks_max_n=KS_MAX_N, random_state=RANDOM_STATE):
    s = pd.Series(series).dropna().astype(float)
    if len(s) < 3:
        return {
            "n": len(s),
            "skew": np.nan,
            "kurtosis": np.nan,
            "shapiro_p": np.nan,
            "ks_p_ref": np.nan,
            "decision": "데이터 부족",
        }

    s_shapiro = s.sample(min(len(s), shapiro_max_n), random_state=random_state) if len(s) > shapiro_max_n else s
    try:
        shapiro_p = stats.shapiro(s_shapiro).pvalue
    except Exception:
        shapiro_p = np.nan

    s_ks = s.sample(min(len(s), ks_max_n), random_state=random_state) if len(s) > ks_max_n else s
    if s_ks.std(ddof=1) == 0 or np.isnan(s_ks.std(ddof=1)):
        ks_p_ref = np.nan
    else:
        z = (s_ks - s_ks.mean()) / s_ks.std(ddof=1)
        ks_p_ref = stats.kstest(z, "norm").pvalue

    skew = s.skew()
    if abs(skew) >= 1 or (pd.notna(shapiro_p) and shapiro_p < 0.05):
        decision = "비모수 권장"
    else:
        decision = "정규 근사 가능"

    return {
        "n": len(s),
        "skew": skew,
        "kurtosis": s.kurtosis(),
        "shapiro_p": shapiro_p,
        "ks_p_ref": ks_p_ref,
        "decision": decision,
    }

normality_df = pd.DataFrame([
    {"variable": c, "description": col_desc.get(c, ""), **normality_summary(infer_df[c])}
    for c in report_cols
]).sort_values(["decision", "skew"], ascending=[True, False])

display(normality_df)


,variable,description,n,skew,kurtosis,shapiro_p,ks_p_ref,decision
5,zone1_resistance,,12931,6.9166,79.5331,0.0000,0.0000,비모수 권장
7,zone1_flow_l_min,1구역 유량,12960,0.9837,-0.1047,0.0000,0.0000,비모수 권장
12,mix_temp_c,양액 온도,12960,0.6491,-0.1253,0.0000,0.0000,비모수 권장
13,mix_flow_l_min,혼합 후 유량,12960,0.4229,-1.4001,0.0000,0.0000,비모수 권장
15,flow_rate_l_min,실제 토출 유량,12960,0.3897,-1.4447,0.0000,0.0000,비모수 권장
8,filter_delta_p_kpa,,12960,0.3424,-1.5375,0.0000,0.0000,비모수 권장
0,motor_current_a,모터 소비 전류,12960,0.1584,-1.7796,0.0000,0.0000,비모수 권장
14,acid_tank_level_pct,산도 조절액 탱크 잔량,12960,0.1416,-1.2941,0.0000,0.0000,비모수 권장
1,motor_power_kw,모터 전력 사용량,12960,0.0983,-1.8560,0.0000,0.0000,비모수 권장
6,zone1_pressure_kpa,1구역 압력,12960,0.0752,-1.8921,0.0000,0.0000,비모수 권장



## 3. 상관 분석 (Pearson / Spearman)
해석 원칙:
- **Spearman**을 주력으로 봅니다. 비선형·비정규에도 비교적 안전합니다.
- **Pearson**은 보조로 봅니다.
- 파생변수와 정의식 구성요소의 상관은 **발견이 아니라 구조적 상관**일 수 있으므로 분리합니다.


In [7]:

def safe_pearson(x, y):
    x = pd.Series(x)
    y = pd.Series(y)
    if x.nunique(dropna=True) < 2 or y.nunique(dropna=True) < 2:
        return np.nan, np.nan
    out = stats.pearsonr(x, y)
    return out.statistic, out.pvalue


def safe_spearman(x, y):
    x = pd.Series(x)
    y = pd.Series(y)
    if x.nunique(dropna=True) < 2 or y.nunique(dropna=True) < 2:
        return np.nan, np.nan
    out = stats.spearmanr(x, y, nan_policy="omit")
    return out.statistic, out.pvalue

corr_rows = []
for target, features in TARGET_DICTIONARY.items():
    if target not in infer_df.columns:
        continue
    for feat in features:
        if feat not in infer_df.columns:
            continue
        pair = infer_df[[target, feat]].dropna()
        if len(pair) < 3:
            continue
        pearson_r, pearson_p = safe_pearson(pair[target], pair[feat])
        spearman_rho, spearman_p = safe_spearman(pair[target], pair[feat])
        corr_rows.append({
            "target": target,
            "target_desc": col_desc.get(target, ""),
            "feature": feat,
            "feature_desc": col_desc.get(feat, ""),
            "n": len(pair),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "relation_type": "구조적 상관" if feat in STRUCTURAL_RELATIONSHIPS.get(target, set()) else "경험적 상관",
        })

corr_df = pd.DataFrame(corr_rows)
display(corr_df.sort_values(["target", "relation_type", "spearman_rho"], ascending=[True, True, False]))


,target,target_desc,feature,feature_desc,n,pearson_r,pearson_p,spearman_rho,spearman_p,relation_type
13,flow_rate_l_min,실제 토출 유량,pump_rpm,펌프 모터 회전 속도,12960,0.9371,0.0000,0.8599,0.0000,경험적 상관
15,flow_rate_l_min,실제 토출 유량,discharge_pressure_kpa,펌프 토출측 압력,12960,0.8714,0.0000,0.6625,0.0000,경험적 상관
16,flow_rate_l_min,실제 토출 유량,filter_delta_p_kpa,,12960,0.7509,0.0000,0.6487,0.0000,경험적 상관
14,flow_rate_l_min,실제 토출 유량,suction_pressure_kpa,펌프 흡입측 압력,12960,-0.8610,0.0000,-0.6697,0.0000,경험적 상관
12,mix_ph,실제 pH 값,acid_tank_level_pct,산도 조절액 탱크 잔량,12960,-0.0023,0.7935,0.0002,0.9815,경험적 상관
8,mix_ph,실제 pH 값,dosing_acid_ml_min,산성액 주입량,12960,0.0011,0.9036,-0.0038,0.6618,경험적 상관
10,mix_ph,실제 pH 값,mix_temp_c,양액 온도,12960,-0.0008,0.9230,-0.0044,0.6189,경험적 상관
11,mix_ph,실제 pH 값,mix_flow_l_min,혼합 후 유량,12960,-0.0005,0.9528,-0.0046,0.5989,경험적 상관
9,mix_ph,실제 pH 값,mix_target_ph,목표 pH 값,12960,NaN,NaN,NaN,NaN,경험적 상관
0,motor_current_a,모터 소비 전류,motor_power_kw,모터 전력 사용량,12960,0.9984,0.0000,0.8792,0.0000,경험적 상관



## 4. 두 집단 / 다중 집단 비교 / 카이제곱
중요:
- 이 섹션은 **외생 그룹 라벨이 실제 CSV에 있을 때만** 실행합니다.
- 지금 CSV에 그룹 라벨이 없으면 자동으로 스킵됩니다.
- 이 설계는 `AE가 만든 라벨로 다시 같은 변수 차이를 검정하는 순환 구조`를 피하기 위한 것입니다.


In [8]:

def cohens_d(x1, x2):
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    n1, n2 = len(x1), len(x2)
    if n1 < 2 or n2 < 2:
        return np.nan
    s1 = x1.std(ddof=1)
    s2 = x2.std(ddof=1)
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / max(n1 + n2 - 2, 1))
    if pooled == 0 or np.isnan(pooled):
        return np.nan
    return (x1.mean() - x2.mean()) / pooled


def rank_biserial_from_u(u_stat, n1, n2):
    if n1 == 0 or n2 == 0:
        return np.nan
    return 1 - (2 * u_stat) / (n1 * n2)


def eta_squared_from_anova(groups):
    all_values = np.concatenate(groups)
    grand_mean = np.mean(all_values)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    ss_total = np.sum((all_values - grand_mean) ** 2)
    return ss_between / ss_total if ss_total != 0 else np.nan


def epsilon_squared_from_kw(h_stat, groups):
    k = len(groups)
    n = sum(len(g) for g in groups)
    return (h_stat - k + 1) / (n - k) if n > k else np.nan

available_group_cols = [c for c in EXTERNAL_GROUP_CANDIDATES if c in infer_df.columns]

binary_groups = []
multiclass_groups = []
for col in available_group_cols:
    n_unique = infer_df[col].dropna().nunique()
    if n_unique == 2:
        binary_groups.append(col)
    elif 3 <= n_unique <= 6:
        multiclass_groups.append(col)

# --- 두 집단 비교 ---
two_group_rows = []
for gcol in binary_groups:
    levels = list(pd.Series(infer_df[gcol].dropna().unique()).tolist())
    if len(levels) != 2:
        continue
    g1, g2 = levels[0], levels[1]
    for target in PRIMARY_TARGETS:
        if target not in infer_df.columns:
            continue
        s1 = infer_df.loc[infer_df[gcol] == g1, target].dropna()
        s2 = infer_df.loc[infer_df[gcol] == g2, target].dropna()
        if len(s1) < MIN_GROUP_N or len(s2) < MIN_GROUP_N:
            continue
        t_out = stats.ttest_ind(s1, s2, equal_var=False, nan_policy="omit")
        u_out = stats.mannwhitneyu(s1, s2, alternative="two-sided")
        two_group_rows.append({
            "group_col": gcol,
            "target": target,
            "n1": len(s1),
            "n2": len(s2),
            "group1": g1,
            "group2": g2,
            "group1_median": s1.median(),
            "group2_median": s2.median(),
            "ttest_p": t_out.pvalue,
            "cohens_d": cohens_d(s1, s2),
            "mwu_p": u_out.pvalue,
            "rank_biserial": rank_biserial_from_u(u_out.statistic, len(s1), len(s2)),
        })

two_group_df = pd.DataFrame(two_group_rows)

# --- 다중 집단 비교 ---
multi_group_rows = []
for gcol in multiclass_groups:
    for target in PRIMARY_TARGETS:
        if target not in infer_df.columns:
            continue
        grouped = [grp[target].dropna().values for _, grp in infer_df.groupby(gcol) if grp[target].dropna().shape[0] >= MIN_GROUP_N]
        labels = [name for name, grp in infer_df.groupby(gcol) if grp[target].dropna().shape[0] >= MIN_GROUP_N]
        if len(grouped) < 3:
            continue
        a_out = stats.f_oneway(*grouped)
        k_out = stats.kruskal(*grouped)
        multi_group_rows.append({
            "group_col": gcol,
            "target": target,
            "k_groups": len(grouped),
            "group_labels": labels,
            "anova_p": a_out.pvalue,
            "eta_squared": eta_squared_from_anova(grouped),
            "kruskal_p": k_out.pvalue,
            "epsilon_squared": epsilon_squared_from_kw(k_out.statistic, grouped),
        })

multi_group_df = pd.DataFrame(multi_group_rows)

# --- 카이제곱 독립성 ---
chi_rows = []
for c1, c2 in combinations(available_group_cols, 2):
    table = pd.crosstab(infer_df[c1], infer_df[c2])
    if table.empty or min(table.shape) < 2:
        continue
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    phi2 = chi2 / max(n, 1)
    r, k = table.shape
    cramers_v = np.sqrt(phi2 / max(min(r - 1, k - 1), 1))
    chi_rows.append({
        "var1": c1,
        "var2": c2,
        "n": n,
        "chi2_p": p,
        "cramers_v": cramers_v,
    })

chi_df = pd.DataFrame(chi_rows)

if not available_group_cols:
    print("외생 그룹 라벨이 현재 CSV에 없어서 두 집단/다중 집단/카이제곱 검정을 스킵합니다.")
else:
    print("[두 집단 비교]")
    display(two_group_df if not two_group_df.empty else pd.DataFrame({"message": ["실행 가능한 이진 그룹 없음"]}))
    print("[다중 집단 비교]")
    display(multi_group_df if not multi_group_df.empty else pd.DataFrame({"message": ["실행 가능한 다중 그룹 없음"]}))
    print("[카이제곱 독립성]")
    display(chi_df if not chi_df.empty else pd.DataFrame({"message": ["실행 가능한 범주형 쌍 없음"]}))


외생 그룹 라벨이 현재 CSV에 없어서 두 집단/다중 집단/카이제곱 검정을 스킵합니다.



## 5. 분포 변화 감지 (KS drift test)
판정 원칙:
- `p < 0.05` 하나만으로 drift라고 쓰지 않습니다.
- **KS statistic**과 **중앙값 변화율**을 같이 봅니다.
- 분류 규칙:
  - `강한 drift`: p<0.05 and ks_stat>=0.10 and median_change>=10%
  - `가능성 있음`: p<0.05 and (ks_stat>=0.05 or median_change>=5%)
  - `통계적 민감 반응`: p<0.05지만 실질 변화는 작음
  - `drift 약함`: 그 외


In [9]:

def drift_summary(df_, columns):
    early = df_.iloc[: len(df_) // 2]
    late = df_.iloc[len(df_) // 2 :]
    rows = []

    for c in columns:
        e = pd.Series(early[c]).dropna().astype(float)
        l = pd.Series(late[c]).dropna().astype(float)
        if len(e) < 3 or len(l) < 3:
            continue

        ks = stats.ks_2samp(e, l, alternative="two-sided", method="auto")
        e_med = e.median()
        l_med = l.median()
        median_change = abs(l_med - e_med) / max(abs(e_med), 1e-9)

        if ks.pvalue < DRIFT_ALPHA and ks.statistic >= DRIFT_STAT_STRONG and median_change >= DRIFT_MEDIAN_STRONG:
            classification = "강한 drift"
        elif ks.pvalue < DRIFT_ALPHA and (ks.statistic >= DRIFT_STAT_WEAK or median_change >= DRIFT_MEDIAN_WEAK):
            classification = "가능성 있음"
        elif ks.pvalue < DRIFT_ALPHA:
            classification = "통계적 민감 반응"
        else:
            classification = "drift 약함"

        rows.append({
            "variable": c,
            "description": col_desc.get(c, ""),
            "n_early": len(e),
            "n_late": len(l),
            "ks_stat": ks.statistic,
            "ks_p": ks.pvalue,
            "early_median": e_med,
            "late_median": l_med,
            "median_change_pct": median_change * 100,
            "classification": classification,
        })

    return pd.DataFrame(rows)


drift_df = drift_summary(infer_df, report_cols).sort_values(["ks_stat", "median_change_pct"], ascending=[False, False])
display(drift_df)


,variable,description,n_early,n_late,ks_stat,ks_p,early_median,late_median,median_change_pct,classification
14,acid_tank_level_pct,산도 조절액 탱크 잔량,6480,6480,0.9704,0.0000,69.8257,45.2832,35.1482,강한 drift
2,motor_temperature_c,모터 온도,6480,6480,0.7843,0.0000,42.9333,47.6732,11.0404,강한 drift
5,zone1_resistance,,6469,6462,0.4792,0.0000,40.4780,67.0262,65.5869,강한 drift
8,filter_delta_p_kpa,,6480,6480,0.4779,0.0000,3.0925,4.7982,55.1576,강한 drift
7,zone1_flow_l_min,1구역 유량,6480,6480,0.4605,0.0000,2.5003,1.1742,53.0347,강한 drift
12,mix_temp_c,양액 온도,6480,6480,0.4585,0.0000,18.9648,18.2225,3.9138,가능성 있음
0,motor_current_a,모터 소비 전류,6480,6480,0.4483,0.0000,3.3290,4.1943,25.9913,강한 drift
4,discharge_pressure_kpa,펌프 토출측 압력,6480,6480,0.4478,0.0000,94.9042,112.1538,18.1757,강한 drift
6,zone1_pressure_kpa,1구역 압력,6480,6480,0.4427,0.0000,70.6132,84.4283,19.5643,강한 drift
1,motor_power_kw,모터 전력 사용량,6480,6480,0.4298,0.0000,1.1375,1.4257,25.3407,강한 drift



## 6. 비판적 리뷰 체크리스트
이 표는 **통계 결과를 그대로 믿지 않기 위한 자기검열 단계**입니다.

판정 규칙:
- `PASS`: 제출 논리상 큰 문제 없음
- `WARN`: 해석 시 주의 문구를 반드시 붙여야 함
- `FAIL`: 지금 상태로 본문 주장에 쓰면 공격받기 쉬움


In [10]:

review_rows = []

review_rows.append({
    "review_item": "외생 그룹 라벨 존재 여부",
    "status": "PASS" if available_group_cols else "FAIL",
    "why": "두 집단/다중 집단 검정은 외생 라벨이 있을 때만 독립 가설검정이 됩니다.",
    "action": "현재 CSV에 그룹 라벨이 없으면 해당 검정 결과는 본문에서 보류",
})

review_rows.append({
    "review_item": "행 단위 독립성",
    "status": "WARN" if autocorr_df["lag1_resampled"].fillna(0).gt(0.8).any() else "PASS",
    "why": "집계 후에도 자기상관이 남아 있으면 p-value가 작게 나올 수 있습니다.",
    "action": "결과 해석 시 '시계열 자기상관이 남아 있음' 문구를 함께 기재",
})

review_rows.append({
    "review_item": "정규성 판정이 K-S 하나에 의존하는가",
    "status": "PASS",
    "why": "이 노트북은 Shapiro와 왜도를 함께 보고, K-S는 참고치로만 사용합니다.",
    "action": "보고서에도 K-S 단독판정 금지 문구 유지",
})

review_rows.append({
    "review_item": "Drift 판정 임계값",
    "status": "PASS",
    "why": "KS p-value뿐 아니라 KS statistic과 중앙값 변화율을 함께 사용합니다.",
    "action": "p<0.05만으로 drift라고 쓰지 않기",
})

structural_exists = bool((corr_df["relation_type"] == "구조적 상관").any()) if not corr_df.empty else False
review_rows.append({
    "review_item": "파생변수 상관 해석",
    "status": "PASS" if structural_exists or corr_df.empty else "WARN",
    "why": "정의식에 포함된 변수와의 상관은 발견이 아니라 구조적 상관일 수 있습니다.",
    "action": "구조적 상관과 경험적 상관을 분리해서 서술",
})

review_df = pd.DataFrame(review_rows)
display(review_df)


,review_item,status,why,action
0,외생 그룹 라벨 존재 여부,FAIL,두 집단/다중 집단 검정은 외생 라벨이 있을 때만 독립 가설검정이 됩니다.,현재 CSV에 그룹 라벨이 없으면 해당 검정 결과는 본문에서 보류
1,행 단위 독립성,WARN,집계 후에도 자기상관이 남아 있으면 p-value가 작게 나올 수 있습니다.,결과 해석 시 '시계열 자기상관이 남아 있음' 문구를 함께 기재
2,정규성 판정이 K-S 하나에 의존하는가,PASS,"이 노트북은 Shapiro와 왜도를 함께 보고, K-S는 참고치로만 사용합니다.",보고서에도 K-S 단독판정 금지 문구 유지
3,Drift 판정 임계값,PASS,KS p-value뿐 아니라 KS statistic과 중앙값 변화율을 함께 사용합니다.,p<0.05만으로 drift라고 쓰지 않기
4,파생변수 상관 해석,PASS,정의식에 포함된 변수와의 상관은 발견이 아니라 구조적 상관일 수 있습니다.,구조적 상관과 경험적 상관을 분리해서 서술



## 7. 제출용 해석 가이드
이 노트북으로 바로 써도 되는 문장과, 아직 쓰면 안 되는 문장을 분리합니다.


In [11]:
submission_guide = pd.DataFrame([
    {
        "category": "바로 본문에 사용 가능",
        "content": "정규성 검정(Shapiro, K-S 참고), 왜도 기반 비모수 권장 여부, Spearman/Pearson 상관, KS drift 결과",
    },
    {
        "category": "주의 문구와 함께 사용",
        "content": "10분 집계 후에도 일부 변수는 자기상관이 남아 있으므로 p-value는 보수적으로 해석해야 함",
    },
    {
        "category": "현재 CSV 기준 본문 보류",
        "content": "외생 그룹 라벨이 없는 상태의 두 집단/다중 집단/카이제곱 검정",
    },
    {
        "category": "해석 시 금지",
        "content": "파생변수와 정의식 구성요소의 상관을 독립적 발견처럼 서술하는 것",
    },
])

display(submission_guide)

print() 
print("권장 보고서 문장 예시:")
print("원시 데이터는 1분 간격 시계열이므로 행 단위 독립성 가정이 약하다. 따라서 기술통계는 raw 기준으로 확인하고, 가설검정은 10분 집계본 기준으로 수행하였다. 정규성은 Shapiro-Wilk, 왜도, K-S 참고값을 함께 검토하였으며, 비정규성이 확인된 변수는 비모수 검정을 우선 해석하였다. Drift는 KS p-value뿐 아니라 KS statistic과 중앙값 변화율을 함께 검토하였다.")


,category,content
0,바로 본문에 사용 가능,"정규성 검정(Shapiro, K-S 참고), 왜도 기반 비모수 권장 여부, Spea..."
1,주의 문구와 함께 사용,10분 집계 후에도 일부 변수는 자기상관이 남아 있으므로 p-value는 보수적으로...
2,현재 CSV 기준 본문 보류,외생 그룹 라벨이 없는 상태의 두 집단/다중 집단/카이제곱 검정
3,해석 시 금지,파생변수와 정의식 구성요소의 상관을 독립적 발견처럼 서술하는 것



권장 보고서 문장 예시:
원시 데이터는 1분 간격 시계열이므로 행 단위 독립성 가정이 약하다. 따라서 기술통계는 raw 기준으로 확인하고, 가설검정은 10분 집계본 기준으로 수행하였다. 정규성은 Shapiro-Wilk, 왜도, K-S 참고값을 함께 검토하였으며, 비정규성이 확인된 변수는 비모수 검정을 우선 해석하였다. Drift는 KS p-value뿐 아니라 KS statistic과 중앙값 변화율을 함께 검토하였다.
